# Route A: an exploratory screen of ploidy, redox and Geminin in cancer cell lines

Choose **Runtime → Run all**. The step-by-step guide (PDF) explains every cell and screen.

In [ ]:
#@title Cell 1 - Setup, settings and connect Google Drive
import io, os, re, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print

# Your data folder in Google Drive (where you put the 7 downloaded files)
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_a_data"  #@param {type:"string"}
DATA_DIR = os.environ.get("ROUTE_A_DATA", DATA_DIR)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


if not os.path.isdir(DATA_DIR):
    raise StopHere(f"Folder not found: {DATA_DIR}. Check the folder name in Google Drive and fix DATA_DIR above.")

# Settings (Colab shows these as boxes on the right)
LINEAGE = "All lineages"  #@param ["All lineages", "CNS/Brain only"]
USE_PROTEINS = True  #@param {type:"boolean"}
EXCLUDE_NRF2_MUTANTS = True  #@param {type:"boolean"}
MY_GENES = ""  #@param {type:"string"}
N_RANDOM_SETS = 500  #@param {type:"integer"}
N_PERMUTATIONS = 2000  #@param {type:"integer"}
MIN_LINES_PER_TISSUE = 10  #@param {type:"integer"}
DEPMAP_RELEASE = "DepMap Public 24Q4"  #@param {type:"string"}
SEED = 1  #@param {type:"integer"}
GENES_OF_INTEREST = "GMNN, CDT1"  #@param {type:"string"}
USE_CRISPR = True  #@param {type:"boolean"}
#@markdown **TP53**: p53 status affects both ploidy tolerance and some redox genes.
TP53_MODE = "All lines"  #@param ["All lines", "TP53-normal lines only", "Adjust for TP53 status"]
#@markdown **Proliferation adjustment**: untick to see how much it changes the results.
ADJUST_PROLIFERATION = True  #@param {type:"boolean"}

# Reporting thresholds (conventions for labelling findings, not hypotheses)
FDR_MAX = 0.05            # false discovery rate
RANDOM_P_MAX = 0.05       # must beat expression-matched random gene sets
CONTROL_P_MAX = 0.05      # must beat every non-redox stress programme

rng = np.random.default_rng(SEED)
GOI = [g.strip().upper() for g in re.split(r"[,\s;]+", GENES_OF_INTEREST) if g.strip()]
print("Genes of interest:", GOI, "| TP53:", TP53_MODE, "| proliferation adjustment:", ADJUST_PROLIFERATION)
print("Data folder:", DATA_DIR)

In [ ]:
#@title Cell 2 - Check the data files in your Drive folder
NEEDED = {
    "models": ["Model.csv"],
    "signatures": ["OmicsSignatures.csv"],
    "expression": ["OmicsExpressionProteinCodingGenesTPMLogp1.csv"],
    "damaging": ["OmicsSomaticMutationsMatrixDamaging.csv"],
    "hotspot": ["OmicsSomaticMutationsMatrixHotspot.csv"],
}
if USE_CRISPR:
    NEEDED["crispr"] = ["CRISPRGeneEffect.csv"]
if USE_PROTEINS:
    NEEDED["proteins"] = ["protein_quant_current_normalized.csv.gz", "protein_quant_current_normalized.csv"]

present = sorted(os.listdir(DATA_DIR))
paths, missing = {}, []
for key, names in NEEDED.items():
    found = next((n for n in names if os.path.exists(os.path.join(DATA_DIR, n))), None)
    if found:
        paths[key] = os.path.join(DATA_DIR, found)
        print(f"   found   {found}  ({os.path.getsize(paths[key]) / 1e6:.0f} MB)")
    else:
        stem = names[0].split(".")[0]
        similar = [p for p in present if p.startswith(stem)]
        missing.append(names[0])
        print(f"   MISSING {names[0]}" + (f"  (similar name in folder: {similar[0]} - rename it)" if similar else ""))
if missing:
    raise StopHere(f"{len(missing)} file(s) missing from {DATA_DIR}. Add or rename them, then run this cell again. "
                   "(Untick USE_CRISPR or USE_PROTEINS in Cell 1 to run without those files.)")
print("All files ready.")

In [ ]:
#@title Cell 3 - Load and match the data
def read_model_matrix(path):
    """Read a DepMap cell-line x gene file (works with old and new layouts)."""
    table = pd.read_csv(path, low_memory=False)
    gene_columns = [c for c in table.columns if re.search(r"\(\d+\)\s*$", str(c))]
    if "ModelID" in table.columns:
        for flag in ("IsDefaultEntryForModel", "is_default_entry"):
            if flag in table.columns:
                table = table[table[flag].astype(str).str.lower().isin(["yes", "true", "1"])]
        table = table.drop_duplicates("ModelID").set_index("ModelID")
    else:
        table = table.set_index(table.columns[0])
    if not gene_columns:
        gene_columns = [c for c in table.columns if pd.api.types.is_numeric_dtype(table[c])]
    table = table[gene_columns].astype("float32")
    table.columns = [re.sub(r"\s*\(\d+\)\s*$", "", str(c)) for c in table.columns]
    table = table.loc[:, ~table.columns.duplicated()]
    table.index = table.index.astype(str)
    return table


def find_column(table, *names):
    lower = {str(c).lower(): c for c in table.columns}
    for name in names:
        if name.lower() in lower:
            return lower[name.lower()]
    raise StopHere(f"Expected a column like {names[0]}; found {list(table.columns)[:12]}")


models = pd.read_csv(paths["models"], low_memory=False)
models = models.set_index(find_column(models, "ModelID"))
models.index = models.index.astype(str)
lineage_col = find_column(models, "OncotreeLineage", "Lineage")
ccle_col = find_column(models, "CCLEName", "CCLE_Name")

signatures = pd.read_csv(paths["signatures"])
signatures = signatures.set_index("ModelID" if "ModelID" in signatures.columns else signatures.columns[0])
signatures.index = signatures.index.astype(str)

print("Reading expression (about 1 minute) ...")
expression = read_model_matrix(paths["expression"])
damaging = read_model_matrix(paths["damaging"])
hotspot = read_model_matrix(paths["hotspot"])


def mutated(matrix, genes):
    present = [g for g in genes if g in matrix.columns]
    return (matrix[present].fillna(0) > 0).any(axis=1) if present else pd.Series(False, index=matrix.index)


nrf2_mutant = pd.concat([mutated(damaging, ["KEAP1", "CUL3"]), mutated(hotspot, ["NFE2L2", "KEAP1"])], axis=1).any(axis=1)
lines = pd.DataFrame({
    "ploidy": pd.to_numeric(signatures[find_column(signatures, "Ploidy")], errors="coerce"),
    "wgd": pd.to_numeric(signatures[find_column(signatures, "WGD")], errors="coerce"),
}).dropna(subset=["ploidy"])
lines = lines[lines.index.isin(expression.index) & lines.index.isin(models.index)]
lines["tissue"] = models.loc[lines.index, lineage_col].astype(str)
lines["nrf2_mutant"] = nrf2_mutant.reindex(lines.index).fillna(False).astype(bool)
tp53_mutant = pd.concat([mutated(damaging, ["TP53"]), mutated(hotspot, ["TP53"])], axis=1).any(axis=1)
lines["tp53_mutant"] = tp53_mutant.reindex(lines.index).fillna(False).astype(bool)
print(f"Cell lines with ploidy and expression: {len(lines)}")
if EXCLUDE_NRF2_MUTANTS:
    print(f"Leaving out {int(lines.nrf2_mutant.sum())} lines with KEAP1, CUL3 or NFE2L2 mutations")
    lines = lines[~lines.nrf2_mutant]
print(f"TP53-mutant lines: {int(lines.tp53_mutant.sum())} of {len(lines)}")
if TP53_MODE == "TP53-normal lines only":
    print(f"Keeping only the {int((~lines.tp53_mutant).sum())} TP53-normal lines")
    lines = lines[~lines.tp53_mutant]
if LINEAGE == "CNS/Brain only":
    lines = lines[lines.tissue == "CNS/Brain"]
sizes = lines.tissue.value_counts()
lines = lines[lines.tissue.isin(sizes[sizes >= MIN_LINES_PER_TISSUE].index)]
print(f"Cell lines analysed: {len(lines)} from {lines.tissue.nunique()} tissues")
if len(lines) < 25:
    raise StopHere(f"Only {len(lines)} cell lines remain. Choose 'All lineages' in Cell 1 and rerun.")

proteins = None
if USE_PROTEINS:
    raw = pd.read_csv(paths["proteins"], low_memory=False)
    gene_col = find_column(raw, "Gene_Symbol", "GeneSymbol")
    sample_cols = [c for c in raw.columns if re.match(r"^.+_TenPx\d+$", str(c))]
    if not sample_cols:
        print("WARNING: protein file layout not recognised; proteins will be skipped.")
    by_gene = raw[[gene_col] + sample_cols].dropna(subset=[gene_col]).groupby(gene_col).mean()
    by_gene.columns = [re.sub(r"_TenPx\d+$", "", c) for c in by_gene.columns]
    by_gene = by_gene.T.groupby(level=0).mean()
    name_to_id = pd.Series(models.index, index=models[ccle_col].astype(str)).groupby(level=0).first()
    by_gene.index = by_gene.index.map(name_to_id)
    proteins = by_gene[by_gene.index.notna() & by_gene.index.isin(lines.index)]
    print(f"Cell lines that also have protein data: {len(proteins)}")

crispr = None
if USE_CRISPR:
    crispr = read_model_matrix(paths["crispr"])
    crispr = crispr[crispr.index.isin(lines.index)]
    print(f"Cell lines that also have CRISPR knockout data: {len(crispr)}")

In [ ]:
#@title Cell 4 - Gene programmes
PROGRAMMES = {
    "REDOX: NRF2 targets": "NQO1 HMOX1 GCLC GCLM TXNRD1 SRXN1 G6PD PGD ME1 TALDO1 TKT FTH1 FTL ABCC1 SLC7A11 GSR PRDX1 TXN SQSTM1 AKR1C1 OSGIN1",
    "REDOX: glutathione": "GSS GSR GPX1 GPX3 GPX4 GPX7 GPX8 GSTM1 GSTM2 GSTM3 GSTP1 GSTO1 GSTA4 GSTK1 MGST1 MGST3 GLRX GLRX2 GLRX3 GCLC GCLM",
    "REDOX: thioredoxin/peroxiredoxin": "TXN TXN2 TXNRD1 TXNRD2 TXNIP PRDX1 PRDX2 PRDX3 PRDX4 PRDX5 PRDX6 SRXN1 NXN TXNDC5 TXNDC12",
    "REDOX: antioxidant enzymes": "SOD1 SOD2 SOD3 CAT GPX1 GPX4 PRDX6 MSRA MSRB2 MSRB3 PARK7 NUDT1 OGG1 MUTYH APEX1",
    "REDOX: ROS sources": "NOX4 NOX1 NOX5 CYBB CYBA NCF1 NCF2 NCF4 DUOX1 DUOX2 XDH AOX1 MAOA MAOB ALOX5 ALOX12 ALOX15",
    "REDOX: ferroptosis": "ACSL4 LPCAT3 TFRC SLC7A11 GPX4 AIFM2 NCOA4 FTH1 FTL ALOX15 SLC3A2 VDAC2 VDAC3 PEBP1 DPP4",
    "REDOX: mitochondrial respiration": "NDUFA1 NDUFA2 NDUFA4 NDUFB1 NDUFB5 NDUFS1 NDUFS2 NDUFV1 SDHA SDHB UQCRC1 UQCRC2 UQCRFS1 CYC1 COX4I1 COX5A COX5B COX6A1 COX7A2 ATP5F1A ATP5F1B ATP5MC1",
    "CONTROL: unfolded protein response": "HSPA5 DDIT3 ATF4 XBP1 HERPUD1 DNAJB9 SEL1L EDEM1 CALR PDIA4 PDIA6 HSP90B1 ATF6 ERN1 EIF2AK3 SDF2L1 CRELD2 MANF",
    "CONTROL: heat shock": "HSPA1A HSPA1B HSPB1 DNAJB1 HSPH1 HSPA8 HSP90AA1 HSP90AB1 HSPE1 HSPD1 BAG3 DNAJA1 HSF1 CRYAB",
    "CONTROL: DNA damage": "CDKN1A GADD45A GADD45B MDM2 BBC3 BAX ATM ATR XPC DDB2 SESN1 RRM2B TP53I3 PPM1D",
    "CONTROL: immediate-early genes": "FOS FOSB JUN JUNB EGR1 ARC NPAS4 NR4A1 DUSP1 IER2 ATF3 KLF4",
}
PROLIFERATION = ("MKI67 TOP2A CCNB1 CDK1 BUB1 AURKA AURKB PLK1 CCNA2 MCM2 MCM3 MCM4 PCNA TYMS RRM2 "
                 "CDC20 KIF11 CENPF NUSAP1 UBE2C BIRC5 TPX2").split()
if MY_GENES.strip():
    PROGRAMMES["REDOX: my genes"] = " ".join(g.upper() for g in re.split(r"[,\s;]+", MY_GENES) if g)
PROGRAMMES = {n: [g for g in dict.fromkeys(v.split()) if g not in PROLIFERATION] for n, v in PROGRAMMES.items()}
print(sum(n.startswith("REDOX") for n in PROGRAMMES), "redox programmes,",
      sum(n.startswith("CONTROL") for n in PROGRAMMES), "stress controls")

In [ ]:
#@title Cell 5 - Scoring and adjustment tools (no results yet)
def zscore(frame):
    return (frame - frame.mean()) / frame.std(ddof=0).replace(0, np.nan)


def programme_scores(data, programmes, min_genes=5):
    """Each programme's score = average z-score of its genes measured in at least 80% of lines."""
    z = zscore(data[data.columns[data.notna().mean() >= 0.8]])
    scores, used = {}, {}
    for name, genes in programmes.items():
        found = [g for g in genes if g in z.columns]
        if len(found) >= min_genes:
            scores[name], used[name] = z[found].mean(axis=1, skipna=True), found
    return pd.DataFrame(scores), used


def adjustment_basis(table):
    """What every test removes first: tissue of origin, and (per Cell 1) proliferation and TP53 status."""
    columns = [pd.get_dummies(table["tissue"]).astype(float).to_numpy()]
    if ADJUST_PROLIFERATION:
        prolif = programme_scores(expression.loc[table.index], {"p": PROLIFERATION})[0]["p"].rank().to_numpy()
        columns.append(((prolif - prolif.mean()) / prolif.std())[:, None])
    if TP53_MODE == "Adjust for TP53 status" and table["tp53_mutant"].nunique() > 1:
        columns.append(table["tp53_mutant"].astype(float).to_numpy()[:, None])
    q, _ = np.linalg.qr(np.column_stack(columns))
    return q


def residual(values, basis):
    values = np.asarray(values, dtype=float)
    return values - basis @ (basis.T @ values)


def corr(a, b):
    a, b = a - a.mean(axis=0), b - b.mean(axis=0)
    return (a * b).sum(axis=0) / np.sqrt((a**2).sum(axis=0) * (b**2).sum(axis=0))


def shuffled_within_tissue(values, groups, n):
    out = np.tile(np.asarray(values, dtype=float)[:, None], (1, n))
    for g in np.unique(groups):
        idx = np.flatnonzero(groups == g)
        for j in range(n):
            out[idx, j] = out[rng.permutation(idx), j]
    return out


def perm_p(observed, null):
    return (1 + np.sum(np.abs(null) >= abs(observed))) / (len(null) + 1)


def fdr(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adjusted = p[order] * len(p) / np.arange(1, len(p) + 1)
    out = np.empty_like(p)
    out[order] = np.minimum(np.minimum.accumulate(adjusted[::-1])[::-1], 1)
    return out
print("Tools ready.")

In [ ]:
#@title Cell 6 - Screen 1: does each programme go with ploidy, within tissue? (RNA)
def test_link(data, label, exposure="ploidy"):
    """Partial rank correlation of programme score with ploidy, after removing tissue and proliferation.
    The p-value comes from shuffling ploidy within each tissue N_PERMUTATIONS times."""
    table = lines[lines.index.isin(data.index)].dropna(subset=[exposure])
    data = data.loc[table.index]
    basis = adjustment_basis(table)
    x = residual(table[exposure].rank().to_numpy(), basis)
    xs = residual(shuffled_within_tissue(table[exposure].rank().to_numpy(), table["tissue"].to_numpy(),
                                         N_PERMUTATIONS), basis)
    scores, used = programme_scores(data, PROGRAMMES)
    rows = []
    for name in scores.columns:
        s = scores[name].to_numpy()
        ok = ~np.isnan(s)
        r = residual(np.where(ok, s, np.nanmean(s)), basis)
        rho = float(corr(x[ok, None], r[ok, None])[0])
        null = corr(xs[ok], np.repeat(r[ok, None], xs.shape[1], axis=1))
        rows.append({"data": label, "programme": name, "lines": int(ok.sum()), "genes_used": len(used[name]),
                     "partial_rho": rho, "p_perm": perm_p(rho, null)})
    result = pd.DataFrame(rows)
    redox = result.programme.str.startswith("REDOX")
    result.loc[redox, "q_fdr"] = fdr(result.loc[redox, "p_perm"])
    return {"result": result, "scores": scores, "used": used, "x": x, "xs": xs, "basis": basis,
            "table": table, "data": data}


rna = test_link(expression, "RNA")
display(rna["result"].round(3))

In [ ]:
#@title Cell 7 - Screen 2: is it stronger than random genes?
def test_random(ctx):
    """For each redox programme, draw N_RANDOM_SETS random gene sets of the same size, each gene matched
    on average expression, and see how strongly they correlate with ploidy."""
    data, used, x, basis, result = ctx["data"], ctx["used"], ctx["x"], ctx["basis"], ctx["result"]
    usable = data.columns[data.notna().mean() >= 0.8]
    level = data[usable].mean()
    bins = pd.qcut(level.rank(method="first"), 20, labels=False)
    avoid = set().union(*used.values()) | set(PROLIFERATION)
    pool = [g for g in usable if g not in avoid]
    by_bin = {b: [g for g in level.index[bins == b] if g not in avoid] or pool for b in range(20)}
    z = zscore(data[usable])
    for name, genes in used.items():
        if not name.startswith("REDOX"):
            continue
        null = []
        for _ in range(N_RANDOM_SETS):
            draw = [rng.choice(by_bin[int(bins[g])]) for g in genes]
            null.append(corr(x[:, None], residual(z[draw].mean(axis=1).fillna(0).to_numpy(), basis)[:, None])[0])
        i = result.programme == name
        rho = float(result.loc[i, "partial_rho"].iloc[0])
        result.loc[i, "random_low"], result.loc[i, "random_high"] = np.quantile(null, [0.025, 0.975])
        result.loc[i, "p_vs_random"] = perm_p(rho, np.array(null))


test_random(rna)
display(rna["result"].loc[rna["result"].programme.str.startswith("REDOX"),
                          ["programme", "partial_rho", "random_low", "random_high", "p_vs_random"]].round(3))

In [ ]:
#@title Cell 8 - Screen 3: is it specific to redox, or general stress?
def test_controls(ctx):
    """For each redox programme and each stress control, test whether (redox minus control) still goes with
    ploidy. The worst (largest) p-value across the four controls is kept."""
    scores, x, xs, basis, result = ctx["scores"], ctx["x"], ctx["xs"], ctx["basis"], ctx["result"]
    controls = [c for c in scores.columns if c.startswith("CONTROL")]
    for name in [n for n in scores.columns if n.startswith("REDOX")]:
        i = result.programme == name
        rho = float(result.loc[i, "partial_rho"].iloc[0])
        worst_p, worst = 0.0, ""
        for control in controls:
            diff = zscore(scores[[name]])[name] - zscore(scores[[control]])[control]
            r = residual(diff.fillna(0).to_numpy(), basis)
            observed = float(corr(x[:, None], r[:, None])[0])
            p = perm_p(observed, corr(xs, np.repeat(r[:, None], xs.shape[1], axis=1)))
            if np.sign(observed) != np.sign(rho):
                p = 1.0
            if p >= worst_p:
                worst_p, worst = p, control.replace("CONTROL: ", "")
        result.loc[i, "p_vs_controls"], result.loc[i, "closest_control"] = worst_p, worst


test_controls(rna)
display(rna["result"][["programme", "partial_rho"]].round(3).join(
    rna["result"][["p_vs_controls", "closest_control"]]))

In [ ]:
#@title Cell 9 - Screen 4: same tests with whole-genome doubling (yes/no)
wgd = test_link(expression, "RNA, whole-genome doubling", exposure="wgd")
test_random(wgd)
test_controls(wgd)
print(f"{int(wgd['table'].wgd.sum())} of {len(wgd['table'])} lines have a doubled genome")
display(wgd["result"].loc[wgd["result"].programme.str.startswith("REDOX"),
                          ["programme", "partial_rho", "q_fdr", "p_vs_random", "p_vs_controls"]].round(3))

In [ ]:
#@title Cell 10 - Screen 5: same tests on proteins
prot = None
if proteins is not None and len(proteins) >= 25:
    prot = test_link(proteins, "Protein")
    test_random(prot)
    test_controls(prot)
    display(prot["result"].loc[prot["result"].programme.str.startswith("REDOX"),
                               ["programme", "lines", "genes_used", "partial_rho", "q_fdr", "p_vs_random",
                                "p_vs_controls"]].round(3))
else:
    print("Proteins skipped: switched off in Cell 1 or fewer than 25 matching cell lines.")

In [ ]:
#@title Cell 11 - Screen 6: gene by gene
def gene_screen(ctx, label):
    """Every redox and control gene, plus GENES_OF_INTEREST, tested on its own against ploidy (same adjustment
    and shuffles as Screen 1). False discovery rate is computed across all genes tested."""
    data, x, xs, basis = ctx["data"], ctx["x"], ctx["xs"], ctx["basis"]
    genes = list(dict.fromkeys([g for v in PROGRAMMES.values() for g in v] + GOI))
    genes = [g for g in genes if g in data.columns and data[g].notna().mean() >= 0.8]
    z = zscore(data[genes])
    R = residual(z.fillna(z.mean()).fillna(0).to_numpy(), basis)
    R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), 1)
    xz = (x - x.mean()) / x.std()
    xsz = (xs - xs.mean(0)) / xs.std(0)
    rho = xz @ R / len(xz)
    null = xsz.T @ R / len(xz)
    p = (1 + (np.abs(null) >= np.abs(rho)).sum(0)) / (null.shape[0] + 1)
    member = {g: ", ".join(n.split(": ")[1] for n, v in PROGRAMMES.items() if g in v) for g in genes}
    out = pd.DataFrame({"data": label, "gene": genes, "partial_rho": rho, "p_perm": p,
                        "programmes": [member[g] or "gene of interest" for g in genes]})
    out["q_fdr"] = fdr(out["p_perm"])
    return out.assign(_abs=out.partial_rho.abs()).sort_values(["p_perm", "_abs"], ascending=[True, False]).drop(columns="_abs")


gene_results = [gene_screen(rna, "RNA")]
if prot is not None:
    gene_results.append(gene_screen(prot, "Protein"))
gene_results = pd.concat(gene_results, ignore_index=True)
both = gene_results.pivot_table(index="gene", columns="data", values=["partial_rho", "q_fdr"])
print("Top 15 genes in RNA:")
display(gene_results[gene_results.data == "RNA"].head(15).round(3))
if prot is not None:
    agree = both[(both[("q_fdr", "RNA")] < FDR_MAX) & (both[("q_fdr", "Protein")] < FDR_MAX) &
                 (np.sign(both[("partial_rho", "RNA")]) == np.sign(both[("partial_rho", "Protein")]))]
    print(f"\nGenes significant in both RNA and protein, same direction: {len(agree)}")
    display(agree.round(3))

In [ ]:
#@title Cell 12 - Screen 7: Geminin and replication licensing
def partial_assoc(table, a, scores, extra=None, n_perm=N_PERMUTATIONS):
    """Partial rank correlation of vector a with each column of scores, within tissue, adjusted for
    proliferation (and optionally ploidy); p-value by shuffling a within tissue."""
    cols = [adjustment_basis(table)]
    if extra is not None:
        cols.append(np.asarray(extra, dtype=float)[:, None])
    basis, _ = np.linalg.qr(np.column_stack(cols))
    a_rank = pd.Series(a).rank().to_numpy()
    xa = residual(a_rank, basis)
    xsa = residual(shuffled_within_tissue(a_rank, table["tissue"].to_numpy(), n_perm), basis)
    rows = []
    for name in scores.columns:
        s = scores[name].to_numpy()
        r = residual(np.where(np.isnan(s), np.nanmean(s), s), basis)
        rho = float(corr(xa[:, None], r[:, None])[0])
        rows.append({"against": name, "partial_rho": rho,
                     "p_perm": perm_p(rho, corr(xsa, np.repeat(r[:, None], xsa.shape[1], axis=1)))})
    out = pd.DataFrame(rows)
    out["q_fdr"] = fdr(out["p_perm"])
    return out


def rho_only(table, A, scores, extra=None):
    """Partial correlations of many vectors (columns of A) with each score, no shuffling (fast)."""
    cols = [adjustment_basis(table)]
    if extra is not None:
        cols.append(np.asarray(extra, dtype=float)[:, None])
    basis, _ = np.linalg.qr(np.column_stack(cols))
    RA = residual(pd.DataFrame(A).rank().to_numpy(), basis)
    S = scores.to_numpy(float)
    RS = residual(np.where(np.isnan(S), np.nanmean(S, axis=0), S), basis)
    RA = (RA - RA.mean(0)) / RA.std(0)
    RS = (RS - RS.mean(0)) / RS.std(0)
    return RA.T @ RS / len(RA)


def matched_genes(gene, table, n=200):
    """Genes like `gene` in average expression and in how closely they follow proliferation."""
    data = expression.loc[table.index]
    avoid = set(g for v in PROGRAMMES.values() for g in v) | set(PROLIFERATION) | set(GOI)
    pool = [g for g in data.columns if g not in avoid and data[g].std() > 0]
    prolif = programme_scores(data, {"p": PROLIFERATION})[0]["p"].rank()
    level = data[pool + [gene]].mean()
    follow = data[pool + [gene]].rank().corrwith(prolif)
    lb = pd.qcut(level.rank(method="first"), 5, labels=False)
    fb = pd.qcut(follow.rank(method="first"), 5, labels=False)
    same = [g for g in pool if lb[g] == lb[gene] and fb[g] == fb[gene]]
    return list(rng.choice(same, min(n, len(same)), replace=False))


table = rna["table"]
measures = {}
for g in GOI:
    if g in expression.columns:
        measures[f"{g} RNA"] = expression.loc[table.index, g].to_numpy(float)
    if crispr is not None and g in crispr.columns:
        measures[f"{g} knockout effect"] = crispr[g].reindex(table.index).to_numpy(float)
if {"GMNN", "CDT1"} <= set(expression.columns):
    measures["GMNN minus CDT1 (RNA)"] = (expression.loc[table.index, "GMNN"] - expression.loc[table.index, "CDT1"]).to_numpy(float)

redox_scores = rna["scores"][[c for c in rna["scores"].columns if c.startswith("REDOX")]]
geminin = []
for label, values in measures.items():
    ok = ~np.isnan(values)
    if ok.sum() < 25:
        continue
    sub = table[ok]
    ploidy_scores = pd.DataFrame({"ploidy": sub["ploidy"].to_numpy(), "whole-genome doubling": sub["wgd"].fillna(0).to_numpy()})
    a = partial_assoc(sub, values[ok], ploidy_scores).assign(measure=label, adjusted_for_ploidy="-")
    b = partial_assoc(sub, values[ok], redox_scores.loc[sub.index].reset_index(drop=True)).assign(measure=label, adjusted_for_ploidy="no")
    c = partial_assoc(sub, values[ok], redox_scores.loc[sub.index].reset_index(drop=True),
                      extra=sub["ploidy"].rank().to_numpy()).assign(measure=label, adjusted_for_ploidy="yes")
    part = pd.concat([a, b, c], ignore_index=True).assign(lines=int(ok.sum()))
    gene = label.split()[0]
    if label.endswith(" RNA") and gene in expression.columns:
        others = matched_genes(gene, sub)
        A = expression.loc[sub.index, others].to_numpy(float)
        targets = pd.concat([ploidy_scores, redox_scores.loc[sub.index].reset_index(drop=True)], axis=1)
        null_no = rho_only(sub, A, targets)
        null_yes = rho_only(sub, A, redox_scores.loc[sub.index].reset_index(drop=True), sub["ploidy"].rank().to_numpy())
        for i, row in part.reset_index(drop=True).iterrows():
            null = (null_yes[:, list(redox_scores.columns).index(row["against"])] if row["adjusted_for_ploidy"] == "yes"
                    else null_no[:, list(targets.columns).index(row["against"])])
            part.loc[part.index[i], "matched_low"] = np.quantile(null, 0.025)
            part.loc[part.index[i], "matched_high"] = np.quantile(null, 0.975)
            part.loc[part.index[i], "p_vs_matched_genes"] = perm_p(row["partial_rho"], null)
    geminin.append(part)
geminin = pd.concat(geminin, ignore_index=True) if geminin else pd.DataFrame()
for c in ("matched_low", "matched_high", "p_vs_matched_genes"):
    if not geminin.empty and c not in geminin.columns:
        geminin[c] = np.nan
if geminin.empty:
    print("None of the genes of interest were found.")
else:
    for label in measures:
        part = geminin[geminin.measure == label]
        if part.empty:
            continue
        print(f"\n{label} ({part.lines.iloc[0]} lines)")
        cols = ["against", "adjusted_for_ploidy", "partial_rho", "p_perm", "q_fdr"]
        cols += [c for c in ("matched_low", "matched_high", "p_vs_matched_genes") if part[c].notna().any()]
        display(part[cols].round(3).reset_index(drop=True))

In [ ]:
#@title Cell 13 - Findings, ranked by strength of evidence
def passes(row):
    return (row["q_fdr"] < FDR_MAX) and (row["p_vs_random"] < RANDOM_P_MAX) and (row["p_vs_controls"] < CONTROL_P_MAX)


results = pd.concat([c["result"] for c in (rna, wgd, prot) if c is not None], ignore_index=True)
redox = results[results.programme.str.startswith("REDOX")].copy()
redox["passes_all_checks"] = redox.apply(passes, axis=1)
findings = []
for name, part in redox.groupby("programme", sort=False):
    get = lambda label: part[part.data == label].iloc[0] if (part.data == label).any() else None
    r, p, w = get("RNA"), get("Protein"), get("RNA, whole-genome doubling")
    passed = [row for row in (r, p) if row is not None and row["passes_all_checks"]]
    signs = {np.sign(row["partial_rho"]) for row in passed}
    if r is not None and p is not None and r["passes_all_checks"] and p["passes_all_checks"] and len(signs) == 1:
        level = "Strong: RNA and protein"
    elif passed:
        level = "Moderate: " + ("RNA" if r is not None and r["passes_all_checks"] else "protein") + " only"
    elif any(row is not None and row["q_fdr"] < FDR_MAX for row in (r, p)):
        level = "Weak: correlates, but not specific"
    else:
        level = "None detected"
    lead = r if r is not None else p
    findings.append({"programme": name[7:], "evidence": level,
                     "direction": "higher with ploidy" if lead["partial_rho"] > 0 else "lower with ploidy",
                     "rho_RNA": None if r is None else round(r["partial_rho"], 3),
                     "rho_protein": None if p is None else round(p["partial_rho"], 3),
                     "whole_genome_doubling": None if w is None else ("passes" if w["passes_all_checks"] else "does not pass"),
                     "closest_control_RNA": None if r is None else r["closest_control"]})
order = {"Strong": 0, "Moderate": 1, "Weak": 2, "None": 3}
findings = pd.DataFrame(findings)
findings = findings.sort_values("evidence", key=lambda s: s.str.split(":").str[0].map(order))
display(findings.reset_index(drop=True))
for _, row in findings.iterrows():
    print(f"{row['programme']}: {row['evidence']}" + ("" if row["evidence"].startswith("None") else f", {row['direction']}"))

In [ ]:
#@title Cell 14 - Charts
plot = results[results.data.isin(["RNA", "Protein"])]
names = list(dict.fromkeys(plot.programme))
fig, ax = plt.subplots(figsize=(8, 0.45 * len(names) + 1.5))
for label, offset, colour in (("RNA", -0.18, "#3b6ea8"), ("Protein", 0.18, "#c26a2e")):
    part = plot[plot.data == label]
    if part.empty:
        continue
    y = np.array([names.index(n) for n in part.programme]) + offset
    ax.scatter(part.partial_rho, y, color=colour, label=label, zorder=3)
    has = part.random_low.notna().to_numpy()
    ax.hlines(y[has], part.random_low[has], part.random_high[has], color="#999999", lw=4, alpha=0.5)
ax.axvline(0, color="#444444", lw=0.8)
ax.set_yticks(range(len(names)))
ax.set_yticklabels([n.replace("REDOX: ", "").replace("CONTROL: ", "control: ") for n in names])
ax.invert_yaxis()
ax.set_xlabel("Partial correlation with ploidy (within tissue, adjusted for proliferation)")
ax.set_title("Programmes vs ploidy. Grey bars: range reached by random gene sets", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "chart_programmes.png"), dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
part = gene_results[gene_results.data == "RNA"]
hit = part.q_fdr < FDR_MAX
ax.scatter(part.partial_rho[~hit], -np.log10(part.q_fdr[~hit]), s=12, color="#bbbbbb")
ax.scatter(part.partial_rho[hit], -np.log10(part.q_fdr[hit]), s=16, color="#3b6ea8")
for _, row in part.head(12).iterrows():
    ax.annotate(row["gene"], (row["partial_rho"], -np.log10(row["q_fdr"])), fontsize=8, xytext=(3, 2),
                textcoords="offset points")
ax.axhline(-np.log10(FDR_MAX), color="#999999", lw=0.8, ls="--")
ax.set_xlabel("Partial correlation with ploidy (RNA)")
ax.set_ylabel("-log10 false discovery rate")
ax.set_title("Gene by gene (Screen 6). Blue: false discovery rate below 0.05", loc="left", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "chart_genes.png"), dpi=150)
plt.show()

In [ ]:
#@title Cell 15 - Save and download the results
# The file names record the settings, so runs with different settings do not overwrite each other
tag = "_".join([{"All lines": "allTP53", "TP53-normal lines only": "TP53normal", "Adjust for TP53 status": "TP53adjusted"}[TP53_MODE],
                "prolif" if ADJUST_PROLIFERATION else "noprolif",
                "brain" if LINEAGE == "CNS/Brain only" else "alltissues"])
outputs = {f"route_a_programmes_{tag}.csv": results, f"route_a_findings_{tag}.csv": findings,
           f"route_a_genes_{tag}.csv": gene_results, f"route_a_geminin_{tag}.csv": geminin}
for name, frame in outputs.items():
    frame.to_csv(os.path.join(DATA_DIR, name), index=False)
print("Saved to your Drive folder:", ", ".join(outputs))
try:
    from google.colab import files
    for name in list(outputs):
        files.download(os.path.join(DATA_DIR, name))
except Exception:
    print("Open the Files panel on the left to download the tables and charts.")